# 10.2 长上下文注意力 (Long Context Attention)

标准注意力的计算复杂度为O(n²)，当序列长度增大时，显存和计算量急剧增长。长上下文注意力机制通过稀疏化、分块、线性化等方式降低复杂度。

## 注意力复杂度问题

| 序列长度 | 标准注意力显存 | FlashAttention显存 | 稀疏注意力显存 |
|----------|---------------|-------------------|----------------|
| 4K | 64MB | 2MB | 8MB |
| 32K | 4GB | 16MB | 64MB |
| 128K | 64GB | 64MB | 256MB |
| 1M | 4TB | 512MB | 2GB |

## 1. 稀疏注意力

稀疏注意力通过限制每个token只关注部分位置，将复杂度从O(n²)降低到O(n√n)或O(n log n)。

### 稀疏注意力模式
- **局部窗口**：每个token只关注前后w个token
- **全局token**：少数全局token可以关注所有位置
- **膨胀注意力**：类似膨胀卷积，间隔采样
- **随机注意力**：随机选择部分远距离位置
- **组合模式**：局部+全局+随机（BigBird风格）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class SparseAttentionMask:
    def __init__(self, seq_len=128, window_size=16, n_global=4, n_random=4):
        self.seq_len = seq_len
        self.window_size = window_size
        self.n_global = n_global
        self.n_random = n_random

    def local_window_mask(self):
        mask = torch.zeros(self.seq_len, self.seq_len)
        for i in range(self.seq_len):
            start = max(0, i - self.window_size // 2)
            end = min(self.seq_len, i + self.window_size // 2 + 1)
            mask[i, start:end] = 1
        return mask

    def global_token_mask(self):
        mask = torch.zeros(self.seq_len, self.seq_len)
        mask[:self.n_global, :] = 1
        mask[:, :self.n_global] = 1
        return mask

    def random_mask(self):
        mask = torch.zeros(self.seq_len, self.seq_len)
        for i in range(self.seq_len):
            random_cols = torch.randperm(self.seq_len)[:self.n_random]
            mask[i, random_cols] = 1
        return mask

    def combined_mask(self):
        local = self.local_window_mask()
        global_m = self.global_token_mask()
        random = self.random_mask()
        causal = torch.tril(torch.ones(self.seq_len, self.seq_len))
        combined = torch.clamp(local + global_m + random, 0, 1) * causal
        return combined

    def sparsity_ratio(self, mask):
        return 1 - mask.sum() / (self.seq_len ** 2)

mask_builder = SparseAttentionMask(seq_len=128, window_size=16, n_global=4, n_random=4)

local = mask_builder.local_window_mask()
global_m = mask_builder.global_token_mask()
combined = mask_builder.combined_mask()

print('=== Sparse Attention Masks ===')
print(f'Sequence length: {mask_builder.seq_len}')
print(f'Local window sparsity: {mask_builder.sparsity_ratio(local):.1%}')
print(f'Global token sparsity: {mask_builder.sparsity_ratio(global_m):.1%}')
print(f'Combined sparsity: {mask_builder.sparsity_ratio(combined):.1%}')
print(f'\nFull attention would need {128*128} entries')
print(f'Combined sparse needs {int(combined.sum())} entries ({mask_builder.sparsity_ratio(combined):.1%} saved)')

print(f'\nKey: Sparse attention reduces computation while preserving key connections.')
print(f'Local window captures nearby context, global tokens capture long-range dependencies.')

## 2. FlashAttention

FlashAttention通过分块计算和重计算策略，将注意力计算的显存复杂度从O(n²)降低到O(n)，同时保持精确计算（非近似）。

### FlashAttention核心思想
1. **分块计算**：将Q、K、V分成小块，逐块计算注意力
2. **在线Softmax**：使用增量式Softmax避免存储完整注意力矩阵
3. **重计算**：反向传播时重新计算注意力，而非存储

### FlashAttention版本
- **FlashAttention-1**：基础版本，2-4x加速
- **FlashAttention-2**：优化GPU利用率，5-8x加速
- **FlashAttention-3**：H100优化，进一步加速

### 工业实践
- 所有主流LLM训练都使用FlashAttention
- PyTorch 2.0+原生支持：`F.scaled_dot_product_attention`
- 不改变计算结果，只是更高效的实现

In [ ]:
class FlashAttentionSimulator(nn.Module):
    def __init__(self, d_model=64, n_heads=4, block_size=32):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.block_size = block_size
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        q = self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)

        scale = 1.0 / math.sqrt(self.d_head)
        Bc = self.block_size
        Br = self.block_size
        O = torch.zeros_like(q)
        l = torch.zeros(B, self.n_heads, T, 1, device=x.device)
        m = torch.full((B, self.n_heads, T, 1), float('-inf'), device=x.device)

        for j in range(0, T, Bc):
            j_end = min(j + Bc, T)
            k_block = k[:, :, j:j_end, :]
            v_block = v[:, :, j:j_end, :]

            for i in range(0, T, Br):
                i_end = min(i + Br, T)
                q_block = q[:, :, i:i_end, :]
                S = q_block @ k_block.transpose(-2, -1) * scale

                m_new = torch.max(m[:, :, i:i_end, :], S.max(dim=-1, keepdim=True).values)
                P = torch.exp(S - m_new)
                l_new = torch.exp(m[:, :, i:i_end, :] - m_new) * l[:, :, i:i_end, :] + P.sum(dim=-1, keepdim=True)

                O[:, :, i:i_end, :] = (O[:, :, i:i_end, :] * torch.exp(m[:, :, i:i_end, :] - m_new) * l[:, :, i:i_end, :]
                                        + P @ v_block) / l_new
                m[:, :, i:i_end, :] = m_new
                l[:, :, i:i_end, :] = l_new

        out = O.transpose(1, 2).reshape(B, T, D)
        return self.out_proj(out)

flash_attn = FlashAttentionSimulator(d_model=64, n_heads=4, block_size=32)
x = torch.randn(2, 64, 64)
out = flash_attn(x)

standard_attn = nn.MultiheadAttention(64, 4, batch_first=True)
standard_out, _ = standard_attn(x, x, x)

print('=== FlashAttention Simulation ===')
print(f'Input: {x.shape}')
print(f'Flash output: {out.shape}')
print(f'Standard output: {standard_out.shape}')

print(f'\nMemory comparison (theoretical):')
for seq_len in [1024, 4096, 16384, 65536]:
    standard_mem = seq_len ** 2 * 4 / (1024**2)
    flash_mem = seq_len * 64 * 4 / (1024**2)
    print(f'  Seq {seq_len:>6d}: Standard={standard_mem:>8.1f}MB, Flash={flash_mem:>8.2f}MB, '
          f'Savings={1-flash_mem/standard_mem:.4%}')

print(f'\nKey: FlashAttention is exact (not approximate), just more memory-efficient.')
print(f'Block size trades off between memory savings and parallelism.')

## 3. 线性注意力与状态空间模型

线性注意力将O(n²)的注意力计算转化为O(n)的递归形式，适合极长序列。

### 核心思想
标准注意力：$O = \text{softmax}(QK^T)V$，需要O(n²)计算

线性注意力：$O = \phi(Q)(\phi(K)^T V)$，先算$K^TV$（O(d²)），再乘$Q$（O(nd)），总计O(nd)

### 代表方法
| 方法 | 复杂度 | 递归 | 特点 |
|------|--------|------|------|
| Linear Attention | O(n) | 是 | 简单但性能下降 |
| Mamba/SSM | O(n) | 是 | 选择性状态空间 |
| RWKV | O(n) | 是 | 线性RNN |
| RetNet | O(n) | 是 | 多尺度保留 |

In [ ]:
class LinearAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.eps = 1e-6

    def feature_map(self, x):
        return F.elu(x) + 1

    def forward(self, x):
        B, T, D = x.shape
        q = self.feature_map(self.q_proj(x)).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.feature_map(self.k_proj(x)).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)

        kv = k.transpose(-2, -1) @ v
        z = q @ k.transpose(-2, -1).sum(dim=-2, keepdim=True).transpose(-2, -1)
        qkv = q @ kv
        out = qkv / (z + self.eps)
        out = out.transpose(1, 2).reshape(B, T, D)
        return self.out_proj(out)

class SelectiveSSM(nn.Module):
    def __init__(self, d_model=64, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.in_proj = nn.Linear(d_model, d_model * 2)
        self.conv1d = nn.Conv1d(d_model, d_model, kernel_size=d_conv, padding=d_conv-1, groups=d_model)
        self.x_proj = nn.Linear(d_model, d_state * 2 + 1)
        self.dt_proj = nn.Linear(1, d_model)
        self.A_log = nn.Parameter(torch.log(torch.randn(d_model, d_state)))
        self.D = nn.Parameter(torch.ones(d_model))
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        xz = self.in_proj(x)
        x_proj, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_proj.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)

        ssm_params = self.x_proj(x_conv)
        B_param = ssm_params[:, :, :self.d_state]
        C_param = ssm_params[:, :, self.d_state:self.d_state*2]
        dt = F.softplus(self.dt_proj(ssm_params[:, :, self.d_state:self.d_state+1]))

        A = -torch.exp(self.A_log)
        y = torch.zeros_like(x_conv)
        h = torch.zeros(B, D, self.d_state, device=x.device)

        for t in range(T):
            dt_t = dt[:, t, :].unsqueeze(-1)
            A_t = A.unsqueeze(0) * dt_t
            B_t = B_param[:, t, :].unsqueeze(1)
            C_t = C_param[:, t, :].unsqueeze(1)
            h = h * torch.exp(A_t) + x_conv[:, t, :].unsqueeze(-1) * B_t
            y[:, t, :] = (h * C_t).sum(dim=-1) + self.D * x_conv[:, t, :]

        y = y * F.silu(z)
        return self.out_proj(y)

x = torch.randn(2, 64, 64)

linear_attn = LinearAttention(d_model=64, n_heads=4)
ssm = SelectiveSSM(d_model=64, d_state=16)

la_out = linear_attn(x)
ssm_out = ssm(x)

print('=== Linear Attention & SSM ===')
print(f'Input: {x.shape}')
print(f'Linear Attention output: {la_out.shape}')
print(f'Selective SSM output: {ssm_out.shape}')

print(f'\nComplexity comparison (d=64, n=seq_len):')
for n in [1024, 4096, 16384, 65536]:
    standard = n * n * 64
    linear = n * 64 * 64
    ssm_ops = n * 64 * 16
    print(f'  n={n:>6d}: Standard={standard:>12d}, Linear={linear:>9d}, SSM={ssm_ops:>9d}')

print(f'\nKey: Linear attention and SSM achieve O(n) complexity.')
print(f'SSM (Mamba) is the most promising for long sequences due to selective mechanism.')
print(f'Hybrid architectures (Attention + SSM) combine the best of both worlds.')

## 4. 长上下文注意力工业实践

### 方法选择
| 序列长度 | 推荐方法 | 原因 |
|----------|----------|------|
| < 32K | FlashAttention | 足够高效，精确计算 |
| 32K-128K | FlashAttention + 稀疏注意力 | 减少显存占用 |
| 128K-1M | 线性注意力/SSM + 局部注意力 | O(n)复杂度 |
| > 1M | SSM + 分层处理 | 递归处理极长序列 |

### 最佳实践
1. **始终使用FlashAttention**：无精度损失，显著加速
2. **混合架构**：底层用SSM，顶层用注意力（Jamba风格）
3. **KV Cache优化**：PagedAttention、MQA/GQA减少KV缓存
4. **梯度检查点**：长序列训练必须使用
5. **Ring Attention**：多GPU分布式处理超长序列